# Reproduce the PESO replication in Colab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/randomwish/cp4285_reccomender_systems/blob/feat/peso-replication/notebooks/03_peso_colab.ipynb)

This notebook re-evaluates our **trained seed-43 weights** for PESO (Yoo et al., ICLR 2026) and its two baselines. It first clones this repository (`REPO` at `BRANCH`, below) for our recorded results, then uses PESO's own evaluation script, unchanged. Run it top to bottom and it prints the Table 2 comparison: our recorded numbers, the numbers you just reproduced, and the paper's. A side note at the end plots our Jacobian-lens probe of the model's middle layers. It does not retrain: training takes about 9 GPU-hours per seed (see `docs/PESO-REPLICATION.md`).

**Before you start**
1. **Runtime → Change runtime type → GPU: A100 or L4.** The model runs in bfloat16, which a T4 cannot run efficiently.
2. **Hugging Face access, for Meta's base model only:**
   - accept Meta's licence on [meta-llama/Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B);
   - add a read token as a Colab secret named `HF_TOKEN` (the key icon in the left sidebar).

   Our trained weights download without login from the GitHub release of `REPO` (`llama-peso-weights-seed43-v1`, 9.7 GB, Llama-3.2-1B-PESO-replication).
3. **Expected time:** the paper's average (blocks 1–3, 5,882 test users per method) takes about 30–60 minutes. Set `FULL = True` for all five blocks (25,166 users per method).

**Testing copy.** The links point at the fork `randomwish/cp4285_reccomender_systems`, branch `feat/peso-replication`. After the merge, set `REPO = "Hong-yiii/cp4285_reccomender_systems"`, `BRANCH = "main"` and the badge to match.

Built with Llama. The weights are a Llama 3.2 derivative, distributed under the Llama 3.2 Community License (`LICENSE.txt` and `NOTICE.txt` in the release).

In [ ]:
# Settings: change only these.
SEED = 43          # the bundled seed (43 ran on Modal A100s)
FULL = False       # False: blocks 1-3, the paper's average; True: all five blocks
REPO = "randomwish/cp4285_reccomender_systems"  # testing fork; the team repo is Hong-yiii/cp4285_reccomender_systems
BRANCH = "feat/peso-replication"                # "main" once merged
RELEASE = f"https://github.com/{REPO}/releases/download/llama-peso-weights-seed43-v1"
UPSTREAM_COMMIT = "edbec5f68ddc83e39f22a041f1f2eaf1c4ad172e"
BASE_REVISION = "4e20de362430cd3b72f300e6b0f18e50e7166e08"

import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "No GPU: switch the runtime to A100 or L4")

## 1. Install the replication's pinned versions

torch 2.5.1 and transformers 4.52.4 match the training runs. Run this cell before anything imports torch; if Colab already imported a different torch, restart the session and rerun from here.

In [ ]:
%pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
%pip install -q transformers==4.52.4 tokenizers==0.21.1 safetensors==0.5.3 huggingface_hub==0.32.3 accelerate==1.7.0
%pip install -q scipy matplotlib  # scipy: PESO's continual_lora.py; matplotlib: the J-lens figure. Colab has both
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.is_available())

## 2. Fetch this repository, PESO's code and data, the base model and our trained weights

This repository is cloned first: it holds the recorded results that step 5 compares against. PESO's repository has no licence file, so it is cloned from its authors at the pinned commit rather than copied. It contains the released Instruments blocks and semantic-ID index. The base model comes from Meta's gated Hugging Face repo, and our weights from this repository's GitHub release. Every release file is checked against its SHA-256, and a rerun skips files that are already verified.

In [ ]:
import hashlib, http.client, json, os, shutil, urllib.request
from pathlib import Path

work = Path("/content" if Path("/content").exists() else Path.cwd() / "peso-colab")
work.mkdir(parents=True, exist_ok=True)
repo = work / "cp4285"
if not repo.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH,
                    f"https://github.com/{REPO}", str(repo)], check=True)

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:
    assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in the environment"

upstream = work / "peso"
if not upstream.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/hsyoo32/peso", str(upstream)], check=True)
subprocess.run(["git", "-C", str(upstream), "checkout", "-q", UPSTREAM_COMMIT], check=True)

from huggingface_hub import snapshot_download
base = snapshot_download("meta-llama/Llama-3.2-1B", revision=BASE_REVISION,
                         allow_patterns=["*.json", "*.safetensors", "tokenizer*"])

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            h.update(chunk)
    return h.hexdigest()

def download(asset, target, attempts=3):
    for _ in range(attempts):
        try:
            with urllib.request.urlopen(f"{RELEASE}/{asset}", timeout=60) as response, open(target, "wb") as f:
                shutil.copyfileobj(response, f, length=1 << 24)
            return
        except (OSError, http.client.HTTPException) as error:  # dropped connection: start the file again
            print(f"retrying {asset}: {error}")
    raise RuntimeError(f"Could not download {RELEASE}/{asset}")

assets = work / "release"
assets.mkdir(exist_ok=True)
download("MANIFEST.json", assets / "MANIFEST.json")
manifest = json.loads((assets / "MANIFEST.json").read_text())
for asset, meta in manifest["assets"].items():
    target = assets / asset
    if not (target.exists() and sha256(target) == meta["sha256"]):
        print(f"downloading {asset} ...", flush=True)
        download(asset, target)
        assert sha256(target) == meta["sha256"], f"Checksum mismatch: {asset}; rerun this cell"
    print(f"verified {asset} ({meta['bytes'] / 1e9:.2f} GB)")

## 3. Rebuild the checkpoint layout PESO's evaluator expects

The release holds only what training changed: the LoRA adapters, with every block's snapshot, and the trained embedding and output layers. PESO's loader builds the frozen base model first and then loads these files with `strict=False`, so each block's state is restored exactly.

Single LoRA and PESO both start from the pretraining stage's block 0, so their block-0 files are rebuilt from it instead of being downloaded twice.

In [ ]:
from safetensors.torch import load_file, save_file

TAGS = {
    "pretrain": ("QUICK_ws20.msl-1.temp0.8.sflora_pretrain.blocks5", "lora_pretrain"),
    "lora": ("QUICK_ws20.msl-1.temp0.8.sflora_cl_0.0.blocks5", "lora"),
    "peso": ("QUICK_ws20.msl-1.temp0.8.sflora_kldiv_latest_cl_2.0.blocks5", "lora_kldiv_latest"),
}
ckpt = work / "ckpt"
shared = [a for a, m in manifest["assets"].items() if "/" not in m["path"]]  # tokenizer, config
for asset, meta in manifest["assets"].items():
    method, _, rest = meta["path"].partition("/")
    if method in TAGS:
        target = ckpt / TAGS[method][0] / rest
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            shutil.copy2(assets / asset, target)
for method, (tag, _) in TAGS.items():
    for asset in shared:
        shutil.copy2(assets / asset, ckpt / tag / manifest["assets"][asset]["path"])
pretrain_block0 = load_file(ckpt / TAGS["pretrain"][0] / "block_0" / "model.safetensors")
subset = {k: v for k, v in pretrain_block0.items() if ".modules_to_save.default." in k}
for path in manifest["derived"]:
    method, _, rest = path.partition("/")
    target = ckpt / TAGS[method][0] / rest
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        save_file(subset, target)
print({tag: sorted(p.name for p in (ckpt / tag).iterdir()) for tag, _ in TAGS.values()})

## 4. Evaluate with PESO's own `continual_test.py`

These are the same settings as the replication: 10 beams, one user at a time, with each block evaluated on the model as it stood after training that block. Batching users is not an option: it changed almost every top-10 list in our tests. The three methods run in parallel on GPUs with at least 40 GB, otherwise one after another (about twice as long).

In [ ]:
COMMON = [
    "--dataset", "Instruments_cl_ori", "--data_path", "../data",
    "--index_file", ".index.epoch20000_edim32_beta0.0.json",
    "--temperature", "0.8", "--max_seq_len", "-1", "--max_his_len", "20", "--num_blocks", "5",
    "--lora_target_modules", "q_proj,v_proj,k_proj,o_proj,gate_proj,down_proj,up_proj",
    "--lora_modules_to_save", "embed_tokens,lm_head",
]
blocks = range(5) if FULL else (1, 2, 3)
results = work / "results"
results.mkdir(exist_ok=True)
env = {**os.environ, "WANDB_MODE": "disabled", "TOKENIZERS_PARALLELISM": "false"}

def evaluate(method, port):
    tag, flag = TAGS[method]
    for block in blocks:
        out = results / f"{method}.block{block}.json"
        if out.exists():
            continue
        cmd = ["torchrun", "--nproc_per_node=1", f"--master_port={port}", "continual_test.py",
               "--ckpt_path", str(ckpt / tag), "--base_model", base, "--seed", str(SEED),
               "--results_file", str(out), "--test_batch_size", "1", "--num_beams", "10",
               "--shift_flag", flag, "--test_block", str(block), *COMMON]
        if method != "pretrain":
            cmd.append("--use_final_model")
        log = results / f"{method}.log"
        print(f"{method}: evaluating block {block} ...", flush=True)
        with log.open("a") as f:
            code = subprocess.run(cmd, cwd=upstream / "Fine-tuning", env=env, stdout=f, stderr=subprocess.STDOUT).returncode
        if code:
            raise RuntimeError(f"{method} block {block} failed; end of {log}:\n{log.read_text()[-3000:]}")

# Each evaluator holds about 7.6 GB; three fit on a 40/80 GB A100, one at a time on a 24 GB L4.
import torch
from concurrent.futures import ThreadPoolExecutor
workers = 3 if torch.cuda.get_device_properties(0).total_memory >= 40e9 else 1
with ThreadPoolExecutor(workers) as pool:
    list(pool.map(evaluate, TAGS, (29600, 29601, 29602)))
print("done:", sorted(p.name for p in results.glob("*.json")))

## 5. Compare with our recorded run and the paper

"Recorded" values are the numbers from our original run on Modal A100s, read from the cloned repository's `replication/peso/results/`. Expect close but not always identical numbers.

- **GPU arithmetic matters.** Upstream's evaluator on an RTX 4090 changed the outcome for 1 (single LoRA) and 4 (PESO) of the 1,772 block-1 users of seed 42, compared with the A100. That is about ±0.0002–0.0005 NDCG@10, the same size as the measured PESO − single LoRA differences.
- **The software environment matters too.** In our own fresh-container test (A100-40GB, Python 3.12, numpy 2), the PESO − single LoRA difference matched exactly (−0.0004), and single LoRA and PESO landed within about 3%. Pretrain came out about 13% lower, up to 24% on block 2. The released weights are verified identical to the trained ones, so treat gaps of that size as environment numerics, not a broken download.
- **Match the hardware.** Use an A100 for the closest reproduction. The paper's numbers are Table 2 (Instruments, averaged over D2–D4), from arXiv 2510.25093v3, p. 8.

In [ ]:
RECORDED = repo / "replication" / "peso" / "results" / f"seed{SEED}" / "results"  # our Modal A100 run
PAPER = {"pretrain": (0.0131, 0.0216), "lora": (0.0150, 0.0253), "peso": (0.0162, 0.0268)}

def metrics(path, block):
    """NDCG@10 and Hit@10 of one block from a continual_test.py results file."""
    m = json.loads(path.read_text())["all_block_results"][f"block_{block}"]["all_prompt_results"][0]
    return m["ndcg@10"], m["hit@10"]

def mean(values):
    return sum(values) / len(values)

now = {(m, b): metrics(results / f"{m}.block{b}.json", b) for m in TAGS for b in blocks}
rec = {(m, b): metrics(RECORDED / f"{TAGS[m][0]}.json", b) for m in TAGS for b in blocks}

print("| Method | Block | NDCG@10 now | recorded | Hit@10 now | recorded |")
print("| --- | --- | --- | --- | --- | --- |")
for (method, block), (n, h) in now.items():
    rn, rh = rec[method, block]
    print(f"| {method} | {block} | {n:.4f} | {rn:.4f} | {h:.4f} | {rh:.4f} |")

print("\nAverage over blocks 1-3 (the paper's D2-D4):")
print("| Method | NDCG@10 now | recorded | paper | Hit@10 now | recorded | paper |")
print("| --- | --- | --- | --- | --- | --- | --- |")
for method in TAGS:
    n, h = (mean([now[method, b][i] for b in (1, 2, 3)]) for i in (0, 1))
    rn, rh = (mean([rec[method, b][i] for b in (1, 2, 3)]) for i in (0, 1))
    print(f"| {method} | {n:.4f} | {rn:.4f} | {PAPER[method][0]:.4f} | {h:.4f} | {rh:.4f} | {PAPER[method][1]:.4f} |")
print("\nPESO - single LoRA, NDCG@10 (blocks 1-3):",
      f"now {mean([now['peso', b][0] - now['lora', b][0] for b in (1, 2, 3)]):+.4f};",
      f"recorded {mean([rec['peso', b][0] - rec['lora', b][0] for b in (1, 2, 3)]):+.4f};",
      f"paper {PAPER['peso'][0] - PAPER['lora'][0]:+.4f}")

## 6. Side note: inside the model (Jacobian lens, seed 42)

**Not recomputed here.** This plots the committed results of our J-lens probe on the seed-42 checkpoints (`replication/peso/results/seed42/reports/jlens.json` and `jlens_items.json`, from `replication/peso/jlens.py` and `jlens_items.py` on an RTX 4090). It needs only the cloned repository, not a GPU.

**How it works.**
- **The lens.** The J-lens (Gurnee et al., 2026) asks a middle layer which item it is preparing to recommend. It passes that layer's state through a linear approximation (the Jacobian) of the remaining layers. The logit lens is the cruder baseline that skips that step.
- **What it reads.** Each item is a 4-part code, like an address: code 1 picks one of 256 broad groups, and codes 2–4 narrow it down to one exact item. The first figure reads code 1 only. The second reads all four codes at every layer, feeding in the correct earlier codes, as the model sees them when it writes the address.
- **The score.** MRR is the mean of 1/rank of the target's first code; chance is 0.024.

In [ ]:
JLENS = json.loads((repo / "replication/peso/results/seed42/reports/jlens.json").read_text())["methods"]
LAYERS = (4, 8, 12)
CHANCE = sum(1 / k for k in range(1, 257)) / 256  # MRR of a random ranking of the 256 first codes
last = JLENS["lora"]["readout_quality"]["state4"]  # single LoRA after the last block; PESO is within 0.001
readout = {lens: [last[f"{lens}_l{layer}"]["mrr"] for layer in LAYERS] for lens in ("jlens", "logit")}
final = last["final"]["mrr"]
drift = {m: {part: [JLENS[m]["drift_split"][f"state{s}"]["l4"][f"{part}_rel"] for s in range(1, 5)]
             for part in ("total", "head", "pathway")} for m in ("lora", "peso")}
agree = [JLENS["lora"]["internal_vs_output"][f"l{layer}"] for layer in LAYERS]

print(f"MRR after the last block (single LoRA): final output {final:.3f}, chance {CHANCE:.3f}")
for lens, values in readout.items():
    print(f"  {lens} lens at layers {LAYERS}: " + ", ".join(f"{v:.3f}" for v in values))
print("Layer-4 lens drift from block 0, states 1-4 (total / item-output layer part / inner-layer part):")
for m, parts in drift.items():
    print(f"  {m}: " + " | ".join(" ".join(f"{v:.2f}" for v in parts[p]) for p in ("total", "head", "pathway")))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ax = axes[0]
ax.plot(LAYERS, readout["jlens"], "o-", label="J-lens")
ax.plot(LAYERS, readout["logit"], "s--", label="logit lens")
ax.axhline(final, color="black", lw=1, label="final output")
ax.axhline(CHANCE, color="grey", ls=":", label="chance")
ax.set(title="1. Reading the coming recommendation", xlabel="layer (of 16)",
       ylabel="MRR of the target's first code", xticks=LAYERS, ylim=(0, 0.3))
ax.legend(loc="upper left", ncol=2, fontsize=8)

ax = axes[1]
states = [1, 2, 3, 4]
for part, label in (("total", "total"), ("head", "item-output layer part"), ("pathway", "inner-layer part")):
    (line,) = ax.plot(states, drift["lora"][part], "o-", label=f"{label}, single LoRA")
    ax.plot(states, drift["peso"][part], "x", ms=11, mew=2, color=line.get_color(), label=f"{label}, PESO")
ax.set(title="2. Layer-4 lens drift from block 0", xlabel="state (after training block)",
       ylabel="change relative to the block-0 lens", xticks=states, ylim=(0, 0.4))
ax.legend(fontsize=8)

ax = axes[2]
x = range(len(LAYERS))
ax.bar([i - 0.2 for i in x], [a["spearman_final_vs_mid"] for a in agree], 0.4,
       label="Spearman: output vs J-lens rank change")
ax.bar([i + 0.2 for i in x], [a["share_output_worse_but_mid_not"] for a in agree], 0.4,
       label="share: output worse, J-lens not")
ax.set(title="3. Do output and internal forgetting move together?", xticks=list(x),
       xticklabels=[f"layer {layer}" for layer in LAYERS], ylim=(0, 1))
ax.legend(fontsize=8)

fig.suptitle("Jacobian-lens probe, seed 42 (blocks 1-3, 500 test users each); PESO's markers sit on single LoRA's lines")
fig.tight_layout()
plt.show()

In [ ]:
ITEMS = json.loads((repo / "replication/peso/results/seed42/reports/jlens_items.json").read_text())["methods"]["lora"]
ALL_LAYERS = list(range(15))  # every layer before the output
CODES = ["code 1 (broad)", "code 2", "code 3", "code 4 (exact)"]
q4 = ITEMS["readout_quality"]["state4"]
item_mrr = [{lens: [q4[f"c{c}"][f"{lens}_l{layer}"]["mrr"] for layer in ALL_LAYERS] for lens in ("jlens", "logit")}
            | {"final": q4[f"c{c}"]["final"]["mrr"]} for c in range(4)]
forget = [ITEMS["forgetting"][f"c{c}"] for c in range(4)]
drift_by_code = [ITEMS["drift_split"]["state4"][f"c{c}"]["l4"] for c in range(4)]
ivo = ITEMS["internal_vs_output"]

print("After the last block, MRR of each code: final output; J-lens at layers 4 / 10 / 14")
for c, name in enumerate(CODES):
    m = item_mrr[c]
    print(f"  {name}: {m['final']:.3f}; {m['jlens'][4]:.3f} / {m['jlens'][10]:.3f} / {m['jlens'][14]:.3f}")
print("From each block's own state to the last (blocks 1-3): share of users whose code rank got worse")
print("  " + ", ".join(f"{name} {f['share_worse']:.0%}" for name, f in zip(CODES, forget)))
print("Layer-4 drift at the last state (item-output layer part / inner-layer part):")
print("  " + ", ".join(f"{name} {d['head_rel']:.2f}/{d['inner_rel']:.2f}" for name, d in zip(CODES, drift_by_code)))
print(f"Output worse but J-lens not, layer 8: {ivo['l8']['share_output_worse_lens_not']:.1%} "
      f"(chance pairing {ivo['l8']['null_mean']:.1%})")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

ax = axes[0][0]
for c, name in enumerate(CODES):
    (line,) = ax.plot(ALL_LAYERS, item_mrr[c]["jlens"], "o-", ms=3, label=f"{name}")
    ax.plot(ALL_LAYERS, item_mrr[c]["logit"], ":", color=line.get_color())
    ax.plot([15.4], [item_mrr[c]["final"]], "*", ms=13, color=line.get_color())
ax.axhline(CHANCE, color="grey", ls=":", lw=1)
ax.set(title="A. Which part of the address each layer can read (J-lens solid, logit lens dotted)",
       xlabel="layer (star = final output)", ylabel="MRR among the level's 256 codes", ylim=(0, 0.8))
ax.legend(fontsize=8, loc="upper left")

ax = axes[0][1]
bars = ax.bar(CODES, [f["share_worse"] for f in forget], color=["tab:red", "tab:grey", "tab:grey", "tab:grey"])
for bar, f in zip(bars, forget):
    ax.annotate(f"mean rank {f['mean_rank_change']:+.1f}", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha="center", va="bottom", fontsize=9)
ax.set(title="B. What forgets: users whose code rank got worse (label: mean rank change, + is worse)",
       ylabel="share of users (blocks 1-3)", ylim=(0, 0.6))

ax = axes[1][0]
x = range(4)
ax.bar([i - 0.2 for i in x], [d["head_rel"] for d in drift_by_code], 0.4, label="item-output layer part")
ax.bar([i + 0.2 for i in x], [d["inner_rel"] for d in drift_by_code], 0.4, label="inner-layer part")
ax.set(title="C. Where the layer-4 lens drifted by the last block", xticks=list(x), xticklabels=CODES,
       ylabel="change relative to the block-0 lens", ylim=(0, 0.6))
ax.legend(fontsize=8)

ax = axes[1][1]
share = [ivo[f"l{layer}"]["share_output_worse_lens_not"] for layer in ALL_LAYERS]
null = [ivo[f"l{layer}"]["null_mean"] for layer in ALL_LAYERS]
sd = [ivo[f"l{layer}"]["null_sd"] for layer in ALL_LAYERS]
rho = [ivo[f"l{layer}"]["spearman_output_vs_lens"] for layer in ALL_LAYERS]
ax.plot(ALL_LAYERS, rho, "s-", ms=3, label="Spearman: output vs J-lens item change")
ax.plot(ALL_LAYERS, share, "o-", ms=3, label="share: output worse, J-lens not")
ax.fill_between(ALL_LAYERS, [n - 2 * s for n, s in zip(null, sd)], [n + 2 * s for n, s in zip(null, sd)],
                color="grey", alpha=0.3, label="same share under chance pairing (±2 sd)")
ax.set(title="D. Hidden knowledge? Internal and output changes go together", xlabel="layer", ylim=(0, 1))
ax.legend(fontsize=8)

fig.suptitle("Item-level J-lens, seed 42, single LoRA (PESO is identical to 3 decimals); blocks 1-3, 500 test users each")
fig.tight_layout()
plt.show()

**What the probe implies**

1. **The model settles the kind of item early and the exact item late.**
   - **Code 1:** at layer 4 the logit lens is at chance, but the J-lens already ranks the target's item group about 4× better than chance (figure 1, panel 1).
   - **Codes 3–4:** the parts that pin down the exact item stay faint until about layer 9, then rise steeply in the last layers to match the output (panel A).
2. **Forgetting sits in the broad choice, not the exact item (panel B).**
   - **The coarse codes slip.** By the last block, 41–43% of users' code-1 and code-2 ranks got worse. Code 1 is the only code that worsens on average, by +0.4 places.
   - **The fine codes improve.** Given the right earlier codes, codes 3–4 *improve* for most users, by −2.6 and −4.5 places.
   - **Why the coarse slip matters.** The model writes the address code by code, keeping only its top candidates at each step, and the target's code 1 sits near rank 15. So a small slip there knocks the item out of the top 10. That fits the 45% churn.
3. **There is no sign of knowledge hidden behind output forgetting (panel D).** Output and internal changes correlate, more strongly toward the top layers. The share of users whose output got worse while the J-lens held (9–18% across layers) is *below* what chance pairing gives (about 21–25%).
4. **Drift is not the same as forgetting (panel C).**
   - **The biggest drift is learning.** Codes 3–4 drift most, almost all inside the network, where LoRA acts, and those codes improve.
   - **The forgetting code tracks the item-output layers.** Code 1, the one that forgets, has the largest item-output layer part, which PESO leaves unpenalized.
   - **Prediction:** a working LoRA-only penalty would mostly restrain learning. Penalizing the item-output layers is the more likely fix for forgetting.
5. **PESO's drift sits exactly on single LoRA's,** for every code and layer, while the lens itself changes clearly from state to state (figure 1, panel 2). This independently confirms that the released penalty (about 10⁻⁶) barely acts.

**Limits:**
- one seed, 500 users per block;
- codes 2–4 are read with the correct earlier codes fed in, which describes the model's choices one step at a time, not its full search;
- the J-lens was validated on much larger models, so on a 1B fine-tuned recommender this is exploratory;
- the inner-layer part also reflects activation changes caused by the embedding updates;
- the drift–forgetting links are correlations.

Details: `docs/PESO-REPLICATION.md` in the repository.